In [ ]:
from machine import ADC
from stepper_motor import step_motor
import time

pot = ADC(0)
motor = step_motor(22, 19, 18, 15)

STOP_LIMIT = 300
UPPER_LIMIT = 65535-50
MAX_STEPS = 50
total_steps = 0

def read_pot():
    return pot.read_u16()

def scale_pot_to_steps(value):
    if value < STOP_LIMIT:
        return 0
    v = min(value - STOP_LIMIT, 65535 - STOP_LIMIT)
    return int((v / (65535 - STOP_LIMIT)) * MAX_STEPS)

while True:
    pot_value = read_pot()
    print("Pot:", pot_value)


    # ---- STOP CONDITIONS ----
    if pot_value < STOP_LIMIT or pot_value > UPPER_LIMIT:
        motor.release()
        print("Motor stopped. Total:", total_steps)
        time.sleep(0.1)
        continue
    
    steps = scale_pot_to_steps(pot_value)

    # ---- DIRECTION ----
    if pot_value < 32768:
        motor.set_direction(-1)   # reverse
    else:
        motor.set_direction(1)    # forward

    # ---- MOVE ----
    for _ in range(steps):
        motor.step()
        total_steps += 1
        time.sleep(0.005)

    print("Moved", steps, "steps, total:", total_steps)
    time.sleep(0.05)